In [1]:
from pathlib import Path

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from core import Config
import pandas as pd
from typing import cast, Sequence
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

config = Config()
sector_numbers: Sequence[int] = range(10, 61, 5)

for i in sector_numbers:
    DATASET: str = f'basis_thresh_50_win_log_sector_{i}-r'
    DATA_PATH: Path = config.training_dir / f'{DATASET}.csv'
    # variables
    DEPTH: int = 4
    ITERATIONS: int = 500
    LEARNING_RATE: float = 0.1
    FOLDS: int = 5
    #ORDERED: str | None = None
    ORDERED: str | None = "Ordered"
    SHUFFLE: bool = True
    STATE: int = 42
    RESULTS_NAME: str = f'OHE_{DEPTH}_{DATASET}'
    if ORDERED:
        RESULTS_NAME += f'_{ORDERED}'
    if SHUFFLE:
        RESULTS_NAME += f'_sh{STATE}'
    SHAP_NAME: str = f'{RESULTS_NAME}_shap_importance.csv'
    SHAP_PLOT_NAME: str = f'{RESULTS_NAME}_shap_beeswarm.png'

    all_data: pd.DataFrame = pd.read_csv(DATA_PATH, index_col=0)

    training_data = all_data.copy()
    training_data.drop(
        training_data[training_data["TR.UpstreamScope3PurchasedGoodsAndServices"].isna()].index,
        inplace=True
    )
    y: pd.DataFrame = training_data['TR.UpstreamScope3PurchasedGoodsAndServices'].to_frame()
    X: pd.DataFrame = training_data.drop('TR.UpstreamScope3PurchasedGoodsAndServices', axis=1)
    # date could also be a categorical variable
    cat_cols: list[str] = ["Instrument", "TR.GICSSectorCode", "TR.HQCountryCode"]
    cat_cols.remove("Instrument")
    # disable categorical features when using one-hot encoded datasets
    #cat_cols: list[str] = []
    X[cat_cols] = X[cat_cols].astype("str")
    gkf: GroupKFold = GroupKFold(n_splits=FOLDS, random_state=STATE, shuffle=SHUFFLE)
    fold_results: list[dict] = []

    best_fold_index: int | None = None
    best_metric_value: float | None = None  # lower is better (RMSE)
    best_model: CatBoostRegressor = CatBoostRegressor()
    best_X_val: pd.DataFrame | None = None
    best_y_val: pd.Series | None = None

    for fold_index, (train_index, val_index) in enumerate(gkf.split(X, y, groups=X['Instrument']), start=1):
        print(f"Fold {fold_index}")
        X_without_instrument = X.drop(columns=["Instrument"])
        X_train: pd.DataFrame = cast(pd.DataFrame, X_without_instrument.iloc[train_index])
        #    X_train: pd.DataFrame = cast(pd.DataFrame, X.iloc[train_index])
        X_val: pd.DataFrame = cast(pd.DataFrame, X_without_instrument.iloc[val_index])
        #    X_val: pd.DataFrame = cast(pd.DataFrame, X.iloc[val_index])
        y_train: pd.Series[float] = cast(pd.Series, y.iloc[train_index])
        y_val: pd.Series[float] = cast(pd.Series, y.iloc[val_index])

        numeric_cols = [c for c in X_train.columns if c not in cat_cols]

        preprocess = ColumnTransformer(
            transformers=[
                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                ("num", "passthrough", numeric_cols),
            ]
        )

        Xt_train = preprocess.fit_transform(X_train)
        Xt_val = preprocess.transform(X_val)

        regressor = CatBoostRegressor(
            iterations=ITERATIONS,
            learning_rate=LEARNING_RATE,
            depth=DEPTH,
            loss_function='RMSE',
            eval_metric='R2',
            use_best_model=True,
            verbose=False,
            early_stopping_rounds=50,
            boosting_type=ORDERED
        )

        ohe_model = regressor

        ohe_model.fit(
            Xt_train, y_train,
            eval_set=(Xt_val, y_val)
        )

        y_val_pred: list[float] = ohe_model.predict(Xt_val)

        rmse: float = float(np.sqrt(mean_squared_error(y_val, y_val_pred)))
        mae: float = cast(float, mean_absolute_error(y_val, y_val_pred))
        r2: float = r2_score(y_val, y_val_pred)

        best_iter = ohe_model.get_best_iteration()

        if best_iter is None:
            print(f"Model ran full {ITERATIONS} iterations (no early stopping).")
            raise Exception("Stopping notebook here")
        elif best_iter + 1 >= ITERATIONS:
            print(f"Model needed all {ITERATIONS} iterations (no earlier convergence).")
            raise Exception("Stopping notebook here")
        else:
            print(f"Model converged earlier at iteration {best_iter + 1} of {ITERATIONS}.")

        fold_results.append(
            {
                "model_name": 'OHE Modell',
                "fold": fold_index,
                "sector": "All",
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
            }
        )

        # update best model based on RMSE
        if (best_metric_value is None) or (rmse < best_metric_value):
            best_metric_value = rmse
            best_fold_index = fold_index
            best_model = ohe_model
            best_X_val = X_val.copy()
            best_y_val = y_val.copy()

        val_sectors: pd.Series = X_val["TR.GICSSectorCode"].astype("category")

        val_sectors = X_val["TR.GICSSectorCode"]
        val_df: pd.DataFrame = pd.DataFrame(
            {
                "y_true": y_val.to_numpy().ravel(),
                "y_pred": y_val_pred,
                "sector": val_sectors.to_numpy().ravel(),
            }
        )

        for sector, grp in val_df.groupby("sector"):
            rmse = float(np.sqrt(mean_squared_error(grp["y_true"], grp["y_pred"])))
            mae = cast(float, mean_absolute_error(grp["y_true"], grp["y_pred"]))
            r2 = r2_score(grp["y_true"], grp["y_pred"])

            fold_results.append(
                {
                    "model_name": 'OHE Modell',
                    "fold": fold_index,
                    "sector": sector,
                    "rmse": rmse,
                    "mae": mae,
                    "r2": r2,
                }
            )

    print(f"Best fold by RMSE: {best_fold_index} (RMSE={best_metric_value:.2f})")
    metrics_df: pd.DataFrame = pd.DataFrame(fold_results)
    metrics_df.to_csv(config.results_dir / f'{RESULTS_NAME}_metrics.csv')

Fold 1
Model converged earlier at iteration 20 of 500.
Fold 2
Model converged earlier at iteration 238 of 500.
Fold 3
Model converged earlier at iteration 19 of 500.
Fold 4
Model converged earlier at iteration 21 of 500.
Fold 5
Model converged earlier at iteration 358 of 500.
Best fold by RMSE: 2 (RMSE=0.97)
Fold 1
Model converged earlier at iteration 30 of 500.
Fold 2
Model converged earlier at iteration 80 of 500.
Fold 3
Model converged earlier at iteration 27 of 500.
Fold 4
Model converged earlier at iteration 97 of 500.
Fold 5
Model converged earlier at iteration 60 of 500.
Best fold by RMSE: 5 (RMSE=1.00)
Fold 1
Model converged earlier at iteration 183 of 500.
Fold 2
Model converged earlier at iteration 92 of 500.
Fold 3
Model converged earlier at iteration 102 of 500.
Fold 4
Model converged earlier at iteration 99 of 500.
Fold 5
Model converged earlier at iteration 279 of 500.
Best fold by RMSE: 1 (RMSE=1.48)
Fold 1
Model converged earlier at iteration 89 of 500.
Fold 2
Model con